Dataset creditcard.csv provided by Kaggle. For secutrity resons, it does not contain the original, but the 28 components of the PCA transformed data. Teh only untransformed columns are Time, Amount and Class

In [1]:
import numpy as np
import pandas as pd

credit_data = pd.read_csv("creditcard.csv")
credit_data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
np.any(credit_data.isna())

False

In [3]:
credit_data["Class"].describe()

count    284807.000000
mean          0.001727
std           0.041527
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: Class, dtype: float64

In [4]:
(np.sum(credit_data["Class"] == 0) / credit_data.shape[0]) * 100

99.82725143693798

Data is highly imbalanced. 99.8% of the datapoints belong to the 'not fraud' class. Just by always predicting a transaction as not fraudulent, our modul would have a 99.8% accuracy and would still be unreliable for the task at hand.

How to handle imbalanced data for a prediciton model? We could:
1. Undersampling: Reduce the number of entries in the majority class
2. Oversampling: Increase the number of entires in the minority class (for example generate new values based on the existing once. simplest approach would be duplication)
3. If possible for the ML model we choose, increase the weight (impact) the minority class has on the algorithm and/or decrease te weight of the majority class

In [5]:
## Make two different versions: One Undersamples, one oversamples
X = credit_data.drop(columns = ["Class"])
y = credit_data.loc[:, "Class"]
print(f"Nr of samples in the unchanged dataset: {X.shape[0]:_}")

## Undersampling
from imblearn.under_sampling import NearMiss
NM = NearMiss()
X_under, y_under = NM.fit_resample(X, y)
print(f"Nr of samples in the undersampled dataset: {X_under.shape[0]:_}")

## Oversampling
from imblearn.over_sampling import SMOTE
SM = SMOTE()
X_over, y_over = SM.fit_resample(X, y)
print(f"Nr of samples in the undersampled dataset: {y_over.shape[0]:_}")

Nr of samples in the unchanged dataset: 284_807
Nr of samples in the undersampled dataset: 984
Nr of samples in the undersampled dataset: 568_630


Steps:
1. Choose Models (k-NN, SVM, RandomForest
2. Build CrossValidation Loop; use crossvall or shufflesplit
3. compare datasets and sampling techniques

In [6]:
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

crossval_dict = {}
crossval_dict["KNC"] = {}
crossval_dict["SVC"] = {}
crossval_dict["RFC"] = {}

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
for name, data_X, data_y in [["Unmodified", X, y], ["Undersampled", X_under, y_under], ["Oversampled", X_over, y_over]]:
    print(name)
    knc_temp = []
    svm_temp = []
    rf_temp = []
    cv = ShuffleSplit(n_splits=10, test_size=0.3, random_state=0)
    for i, (train_index, val_index) in enumerate(cv.split(data_X)):
        print(i+1)
        X_train, X_val = data_X.iloc[train_index, :], data_X.iloc[val_index, :]
        y_train, y_val = data_y.iloc[train_index], data_y.iloc[val_index]

        print("KNN")
        model = KNeighborsClassifier(weights = "distance", n_neighbors=20)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        acc = np.mean(y_pred == y_val.to_numpy().flatten())
        knc_temp.append(acc)

        print("SVC")
        model = SVC(kernel = "linear", C = 2)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        acc = np.mean(y_pred == y_val.to_numpy().flatten())
        svm_temp.append(acc)

        print("RFC")
        if name == "Oversampled":
            #### RFC takes to long on my personal computer with the oversampled dataset
            rf_temp.append(np.nan)
        else:
            model = RandomForestClassifier(min_samples_leaf = 50, min_impurity_decrease = 1e-4)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            acc = np.mean(y_pred == y_val.to_numpy().flatten())
            rf_temp.append(acc)

    crossval_dict["KNC"][name] = np.median(knc_temp)
    crossval_dict["SVC"][name] = np.median(svm_temp)
    crossval_dict["RFC"][name] = np.median(rf_temp)
    print("\n----------------------\n")

Unmodified
1
KNN
SVC


In [ ]:
y_pred

In [ ]:
y_val